# TinySLM — F Long Train TS-01 / Kaggle v01

Continuação do **F_h512_L12 (~51.3M parâmetros)** a partir do checkpoint de **60M tokens**, até aproximadamente **1 passagem pelo TS-01 limpo (~473.13M tokens)**.

### MVP
- Input único: `grid-tiny-v1`
- Carrega `F_h512_L12_60M.zip`
- Reconstrói o fluxo packed determinístico `história → EOS → história → EOS`
- Salta os primeiros 60M tokens já vistos
- Continua até ao fim da primeira passagem pelo corpus
- Avalia e gera exemplos em **120M, 240M, 360M e final (~473M)**
- Guarda **modelo + optimizer + scheduler + posição**, permitindo retoma dentro de outputs persistidos

> Esta é uma nova fase de otimização: os pesos vêm dos 60M, mas optimizer/scheduler começam de novo.


In [ ]:
import os, json, math, time, zipfile, gc, random
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, LlamaForCausalLM
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

SEED=42
CONTEXT=512
START_TOTAL_TOKENS=60_000_256
EXPECTED_CORPUS_TOKENS=473_132_847
TARGETS=[120_000_000,240_000_000,360_000_000,EXPECTED_CORPUS_TOKENS]

BATCH_SIZE=8
GRAD_ACCUM_STEPS=1
LR=2e-4
WEIGHT_DECAY=0.1
WARMUP_RATIO=0.02
MAX_GRAD_NORM=1.0
EVAL_BATCHES=100

RUN_TRAINING=False

INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working/tinyslm_F_long_v01')
WORK.mkdir(parents=True,exist_ok=True)

print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
print('RUN_TRAINING:',RUN_TRAINING)


## 1. Localizar corpus, tokenizer e checkpoint F@60M

In [ ]:
files=[p for p in INPUT.rglob('*') if p.is_file()]
print('Ficheiros encontrados:',len(files))
for p in files: print(p)

def one(name):
    xs=[p for p in files if p.name==name]
    if not xs: raise FileNotFoundError(name)
    return xs[0]

TRAIN=one('train.jsonl')
VAL=one('validation.jsonl')
MODEL_ZIP=one('F_h512_L12_60M.zip')
tokjson=one('tokenizer.json')
TOKDIR=tokjson.parent

MODEL_DIR=WORK/'F_60M'
if not (MODEL_DIR/'model.safetensors').exists():
    MODEL_DIR.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(MODEL_ZIP) as z: z.extractall(MODEL_DIR)

print('TRAIN:',TRAIN)
print('VAL:',VAL)
print('TOKENIZER:',TOKDIR)
print('CHECKPOINT:',MODEL_ZIP)
print('MODEL_DIR:',MODEL_DIR)


## 2. Validar checkpoint e tokenizer

In [ ]:
tok=AutoTokenizer.from_pretrained(str(TOKDIR),local_files_only=True)
assert len(tok)==8000 and tok.pad_token_id==0 and tok.bos_token_id==1 and tok.eos_token_id==2

model_cfg=json.loads((MODEL_DIR/'config.json').read_text())
print('Checkpoint architecture:')
for k in ['hidden_size','num_hidden_layers','num_attention_heads','intermediate_size','vocab_size']:
    print(k,model_cfg.get(k))

assert model_cfg['hidden_size']==512
assert model_cfg['num_hidden_layers']==12
assert model_cfg['num_attention_heads']==8
assert model_cfg['intermediate_size']==2048
assert model_cfg['vocab_size']==8000
print('Checkpoint F e tokenizer OK.')


## 3. Construir fluxo packed completo

O corpus limpo tinha sido auditado em **473,132,847 tokens incluindo um EOS por história**. Construímos o fluxo completo como `uint16`.

O último bloco é preenchido apenas para facilitar batching; os tokens de padding **não contam** para o total da experiência e serão ignorados na loss final se necessário.


In [ ]:
N_BLOCKS=math.ceil(EXPECTED_CORPUS_TOKENS/CONTEXT)
PADDED_TOTAL=N_BLOCKS*CONTEXT
FULL_PACK=WORK/'ts01_full_epoch.uint16'

if not (FULL_PACK.exists() and FULL_PACK.stat().st_size==PADDED_TOTAL*2):
    a=np.memmap(FULL_PACK,dtype=np.uint16,mode='w+',shape=(PADDED_TOTAL,))
    pos=0
    with TRAIN.open('r',encoding='utf-8') as f:
        for i,line in enumerate(f,1):
            text=json.loads(line)['text']
            ids=tok.encode(text,add_special_tokens=False)+[tok.eos_token_id]
            take=min(len(ids),EXPECTED_CORPUS_TOKENS-pos)
            if take:
                a[pos:pos+take]=np.asarray(ids[:take],dtype=np.uint16); pos+=take
            if i%100_000==0: print(f'{i:,} histórias | {pos:,}/{EXPECTED_CORPUS_TOKENS:,}')
            if pos>=EXPECTED_CORPUS_TOKENS: break
    assert pos==EXPECTED_CORPUS_TOKENS,(pos,EXPECTED_CORPUS_TOKENS)
    if PADDED_TOTAL>EXPECTED_CORPUS_TOKENS:
        a[EXPECTED_CORPUS_TOKENS:]=tok.pad_token_id
    a.flush()

print('Corpus real:',f'{EXPECTED_CORPUS_TOKENS:,}')
print('Packed físico:',f'{PADDED_TOTAL:,}')
print('Padding final:',PADDED_TOTAL-EXPECTED_CORPUS_TOKENS)


## 4. Validation packed

In [ ]:
VAL_PACK=WORK/'validation.uint16'
VAL_META=WORK/'validation_meta.json'
if not (VAL_PACK.exists() and VAL_META.exists()):
    ids=[]
    with VAL.open('r',encoding='utf-8') as f:
        for line in f:
            text=json.loads(line)['text']
            ids.extend(tok.encode(text,add_special_tokens=False)); ids.append(tok.eos_token_id)
    usable=len(ids)//CONTEXT*CONTEXT
    np.asarray(ids[:usable],dtype=np.uint16).tofile(VAL_PACK)
    VAL_META.write_text(json.dumps({'raw':len(ids),'usable':usable}))
vm=json.loads(VAL_META.read_text())
print(vm)


## 5. Dataset da continuação

Começamos no primeiro bloco completo após os **60,000,256 tokens** já vistos na Grid A. Esse valor é divisível por 512, portanto não há ambiguidade na fronteira.


In [ ]:
assert START_TOTAL_TOKENS%CONTEXT==0
START_BLOCK=START_TOTAL_TOKENS//CONTEXT
END_BLOCK=math.ceil(EXPECTED_CORPUS_TOKENS/CONTEXT)

class RangeBlocks(Dataset):
    def __init__(self,path,start_block,end_block,real_total=None):
        self.a=np.memmap(path,dtype=np.uint16,mode='r')
        self.start=start_block; self.end=end_block; self.real_total=real_total
    def __len__(self): return self.end-self.start
    def __getitem__(self,i):
        b=self.start+i
        arr=np.array(self.a[b*CONTEXT:(b+1)*CONTEXT],dtype=np.int64)
        labels=arr.copy()
        if self.real_total is not None and (b+1)*CONTEXT>self.real_total:
            valid=max(0,self.real_total-b*CONTEXT)
            labels[valid:]=-100
        return {'input_ids':torch.from_numpy(arr),'labels':torch.from_numpy(labels)}

train_ds=RangeBlocks(FULL_PACK,START_BLOCK,END_BLOCK,EXPECTED_CORPUS_TOKENS)
val_blocks=vm['usable']//CONTEXT
val_ds=RangeBlocks(VAL_PACK,0,val_blocks,None)

remaining=EXPECTED_CORPUS_TOKENS-START_TOTAL_TOKENS
print('Start block:',f'{START_BLOCK:,}')
print('Remaining blocks:',f'{len(train_ds):,}')
print('Tokens reais restantes:',f'{remaining:,}')


## 6. Avaliação qualitativa e quantitativa

In [ ]:
PROMPTS=[
"Once upon a time, there was a little girl named Lily who",
"Tom was afraid of the dark, but one night",
"The dog could not find its ball, so",
"Anna opened the old box and was surprised because",
"Jack promised his mother that he would help her, but",
]
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'

@torch.no_grad()
def evaluate(model):
    model.eval(); losses=[]
    for i,b in enumerate(DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False)):
        if i>=EVAL_BATCHES: break
        x=b['input_ids'].to(DEVICE); y=b['labels'].to(DEVICE)
        with torch.autocast('cuda',dtype=torch.float16,enabled=DEVICE=='cuda'):
            loss=model(input_ids=x,labels=y).loss
        losses.append(loss.item())
    L=sum(losses)/len(losses); model.train()
    return L,math.exp(min(L,20))

@torch.no_grad()
def generate_samples(model):
    model.eval(); old=model.config.use_cache; model.config.use_cache=True
    out=[]
    for p in PROMPTS:
        x=tok(p,return_tensors='pt',add_special_tokens=False).to(DEVICE)
        y=model.generate(**x,max_new_tokens=120,do_sample=True,temperature=.8,top_p=.95,
                         eos_token_id=2,pad_token_id=0)
        out.append({'prompt':p,'text':tok.decode(y[0],skip_special_tokens=True)})
    model.config.use_cache=old; model.train()
    return out


## 7. Treino longo F

Os checkpoints guardam:
- pesos + tokenizer;
- optimizer;
- scheduler;
- scaler;
- `total_tokens_seen`;
- índice do próximo bloco;
- métricas e gerações.

Isto torna os checkpoints aptos a uma continuação posterior.


In [ ]:
REAL_REMAINING=EXPECTED_CORPUS_TOKENS-START_TOTAL_TOKENS
TOKENS_PER_STEP=BATCH_SIZE*CONTEXT*GRAD_ACCUM_STEPS
TOTAL_OPT_STEPS=math.ceil(REAL_REMAINING/TOKENS_PER_STEP)
WARMUP_STEPS=max(1,int(TOTAL_OPT_STEPS*WARMUP_RATIO))

def lrfun(step):
    if step<WARMUP_STEPS: return step/max(1,WARMUP_STEPS)
    p=(step-WARMUP_STEPS)/max(1,TOTAL_OPT_STEPS-WARMUP_STEPS)
    return .5*(1+math.cos(math.pi*min(1,p)))

def save_long(model,opt,sch,scaler,total_seen,next_block,metrics,label):
    d=WORK/'checkpoints'/label; d.mkdir(parents=True,exist_ok=True)
    model.save_pretrained(d,safe_serialization=True); tok.save_pretrained(d)
    torch.save({
        'optimizer':opt.state_dict(),'scheduler':sch.state_dict(),'scaler':scaler.state_dict(),
        'total_tokens_seen':total_seen,'next_block':next_block,
        'metrics':metrics
    },d/'training_state.pt')
    (d/'metrics.json').write_text(json.dumps(metrics,indent=2,ensure_ascii=False))
    print('Checkpoint:',d)

def run_long():
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model=LlamaForCausalLM.from_pretrained(str(MODEL_DIR),local_files_only=True,
                                           torch_dtype=torch.float16).to(DEVICE)
    model.config.use_cache=False
    opt=AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    sch=LambdaLR(opt,lrfun)
    scaler=torch.amp.GradScaler('cuda',enabled=DEVICE=='cuda')

    dl=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=False,pin_memory=True)
    total_seen=START_TOTAL_TOKENS
    target_i=0
    opt.zero_grad(set_to_none=True); model.train(); start=time.time()

    for batch_i,b in enumerate(dl):
        x=b['input_ids'].to(DEVICE,non_blocking=True)
        y=b['labels'].to(DEVICE,non_blocking=True)
        valid=(y!=-100).sum().item()

        with torch.autocast('cuda',dtype=torch.float16,enabled=DEVICE=='cuda'):
            loss=model(input_ids=x,labels=y).loss/GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()

        if (batch_i+1)%GRAD_ACCUM_STEPS==0 or batch_i==len(dl)-1:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),MAX_GRAD_NORM)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sch.step()

        total_seen+=valid
        while target_i<len(TARGETS) and total_seen>=TARGETS[target_i]:
            target=TARGETS[target_i]
            vl,ppl=evaluate(model)
            met={'total_tokens_seen':total_seen,'target_tokens':target,
                 'validation_loss':vl,'perplexity':ppl,
                 'train_loss':float(loss.detach().cpu())*GRAD_ACCUM_STEPS,
                 'lr':sch.get_last_lr()[0],'elapsed_hours':(time.time()-start)/3600,
                 'samples':generate_samples(model)}
            label='final_473M' if target==EXPECTED_CORPUS_TOKENS else f'{target//1_000_000}M'
            next_block=START_BLOCK+(batch_i+1)*BATCH_SIZE
            save_long(model,opt,sch,scaler,total_seen,next_block,met,label)
            print(f"{label} | val={vl:.4f} | ppl={ppl:.2f} | {met['elapsed_hours']:.2f} h")
            target_i+=1

        if total_seen>=EXPECTED_CORPUS_TOKENS: break

    return model


## 8. Gate

In [ ]:
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NÃO')
print('Checkpoint inicial:',MODEL_DIR)
print('Começa em:',f'{START_TOTAL_TOKENS:,}','tokens')
print('Termina em:',f'{EXPECTED_CORPUS_TOKENS:,}','tokens')
print('Restantes:',f'{REAL_REMAINING:,}','tokens')
print('Targets:',TARGETS)
print('LR nova fase:',LR)
print('RUN_TRAINING:',RUN_TRAINING)


## 9. Executar

Para o run real/agendado, mude `RUN_TRAINING=True`, faça **Save Version / Run All** ou configure o Schedule conforme a interface do Kaggle.


In [ ]:
if RUN_TRAINING:
    if not torch.cuda.is_available(): raise RuntimeError('GPU não disponível')
    final_model=run_long()
else:
    print('BLOQUEADO — preparação validada; treino longo não iniciado.')


## 10. Ver textos dos checkpoints concluídos

In [ ]:
ckroot=WORK/'checkpoints'
if ckroot.exists():
    for d in sorted([x for x in ckroot.iterdir() if x.is_dir()]):
        mf=d/'metrics.json'
        if not mf.exists(): continue
        m=json.loads(mf.read_text())
        print('\n'+'='*100)
        print(d.name,'| loss',round(m['validation_loss'],4),'| ppl',round(m['perplexity'],2))
        print('='*100)
        for s in m['samples']:
            print('\nPROMPT:',s['prompt'])
            print(s['text'])
else:
    print('Ainda não existem checkpoints.')
